# RL for Qubit Rotation

You will solve the **same** control task (single-qubit rotation) using **three** approaches:

1. **Random policy** (baseline)
2. **Policy gradient (REINFORCE)** — learn a *stochastic policy* directly
3. **Q-learning (DQN)** — learn an *action-value function* and act greedily


---

## Physics model (simple and analytic)

We restrict to a great circle on the Bloch sphere:

$
|\psi(\theta)\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle,
\quad \theta \in [0,\pi].
$

- Start: $(\theta_0 = 0)$ (state $(|0\rangle)$
- Target: $(\theta^\star=\pi)$ (state $(|1\rangle)$)

Observation to the agent:
$
s(\theta) = (\cos\theta, \sin\theta).
$

Actions (discrete controls): choose $(\Delta\theta)$ from a small set.

---

## Fidelity and reward shaping

Fidelity to target $|1\rangle$:

$
F(\theta) = |\langle 1|\psi(\theta)\rangle|^2 = \sin^2(\theta/2).
$

Dense shaping reward:

$
r_t^{\text{shape}} = F(\theta_{t+1}) - F(\theta_t).
$

Terminal success bonus:
- If $F(\theta_{t+1}) \ge F_{\text{success}}$, then **add +1.0** and terminate.

**Why add +1.0?**  
Because success is a discrete event (episode termination). The bonus makes successful episodes clearly stand out, improving the learning signal (especially for policy gradients) and discouraging “almost-success” behavior that hovers near the threshold.

---


Search for `TODO` cells and fill them in.


## Notebook structure

For each method, you will follow the same steps:

1. **Define the policy / agent**
2. **Run rollouts (episodes)**
3. **Compute an update (if learning)**
4. **Evaluate + visualize**

You can compare methods using:
- success rate
- average final fidelity
- average episode length


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import matplotlib.pyplot as plt
from collections import deque

def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(0)
print("Device:", device)


## Environment (shared)

In [ ]:
class QubitRotationEnv:
    # Single-qubit rotation environment for RL comparisons.
    #
    # State: theta in [0, pi]
    # Observation: [cos(theta), sin(theta)]
    # Discrete actions: delta-theta values
    # Reward: fidelity improvement + terminal bonus

    def __init__(self, deltas=None, max_steps=40, success_F=0.999):
        if deltas is None:
            deltas = [-0.3, -0.15, 0.0, 0.15, 0.3]  # radians
        self.deltas = np.array(deltas, dtype=np.float32)
        self.n_actions = len(self.deltas)
        self.max_steps = int(max_steps)
        self.success_F = float(success_F)
        self.theta = None
        self.t = 0

    @staticmethod
    def fidelity(theta: float) -> float:
        return float(np.sin(theta/2.0)**2)

    @staticmethod
    def obs(theta: float) -> np.ndarray:
        return np.array([np.cos(theta), np.sin(theta)], dtype=np.float32)

    def reset(self):
        self.theta = 0.0
        self.t = 0
        return self.obs(self.theta)

    def step(self, action: int):
        a = int(action)
        delta = float(self.deltas[a])
        F_old = self.fidelity(self.theta)

        self.theta = float(np.clip(self.theta + delta, 0.0, float(np.pi)))
        self.t += 1

        F_new = self.fidelity(self.theta)
        reward = (F_new - F_old)

        done = False
        if F_new >= self.success_F:
            reward += 1.0
            done = True
        if self.t >= self.max_steps:
            done = True

        info = {"theta": self.theta, "fidelity": F_new}
        return self.obs(self.theta), float(reward), bool(done), info


## Common evaluation helpers

We will evaluate a policy by running multiple episodes and computing:
- success rate
- mean final fidelity
- mean episode length


In [ ]:
def run_episodes(env, act_fn, n_episodes=200):
    """Run episodes with a provided action function: a = act_fn(obs)."""
    successes = 0
    final_F = []
    lengths = []
    returns = []

    for _ in range(n_episodes):
        obs = env.reset()
        total_r = 0.0
        info = {"fidelity": env.fidelity(0.0)}
        for t in range(env.max_steps):
            action = act_fn(obs)  # int
            obs, r, done, info = env.step(action)
            total_r += r
            if done:
                break
        lengths.append(t+1)
        returns.append(total_r)
        final_F.append(info["fidelity"])
        if info["fidelity"] >= env.success_F:
            successes += 1

    return {
        "success_rate": successes / n_episodes,
        "mean_final_fidelity": float(np.mean(final_F)),
        "mean_length": float(np.mean(lengths)),
        "mean_return": float(np.mean(returns)),
    }

def print_metrics(name, m):
    print(f"{name:>18s} | success={m['success_rate']:.2f} | final F={m['mean_final_fidelity']:.4f} | len={m['mean_length']:.1f} | return={m['mean_return']:.3f}")


# Part A — Random policy (baseline)

A random policy samples actions uniformly:

$
\pi(a\mid s)=\frac{1}{|\mathcal{A}|}.
$

This is not expected to solve the task reliably, but it gives a **baseline**.

**TODO:** implement a random action function and evaluate it.


In [ ]:
env = QubitRotationEnv(max_steps=40)
nA = env.n_actions

def act_random(obs):
    # TODO: return a random integer action in {0,1,...,nA-1}
    raise NotImplementedError

m_rand = run_episodes(env, act_random, n_episodes=300)
print_metrics("Random", m_rand)


# Part B — Policy Gradient (REINFORCE)

## What is learned?
A neural network policy \( \pi_\theta(a\mid s) \) outputting action probabilities.

## Update rule
Discounted return:
$
G_t = \sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.
$

REINFORCE gradient estimator:
$
\nabla_\theta J(\theta) \approx \sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)\, \hat G_t.
$

Implemented as minimizing:
$
\mathcal{L}(\theta) = -\sum_t \log \pi_\theta(a_t\mid s_t)\, \hat G_t.
$

### Variance reduction
Normalize returns:
$
\hat G_t \leftarrow \frac{G_t - \mu}{\sigma + \epsilon}.
$

---


In [ ]:
class PolicyNet(nn.Module):
    """
    Policy network for REINFORCE.

    Input:
        observation s = (cos(theta), sin(theta))  ∈ R^2

    Output:
        logits for each discrete action a

    The softmax will be applied in the forward pass to obtain
    a probability distribution π(a|s).
    """
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()

        # TODO:
        # Build a small feed-forward neural network using nn.Sequential.
        # You may experiment with depth or width later.
        self.net = nn.Sequential(
            # TODO: nn.Linear(...)
            # TODO: nn.ReLU()
            # TODO: nn.Linear(...)
            # TODO: nn.ReLU()
            # TODO: nn.Linear(...)
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Returns:
            Action probabilities π(a|s) as a 1D tensor of size n_actions.
        """
        logits = self.net(obs)

        # TODO:
        # Convert logits into a probability distribution over actions.
        # Hint: torch.softmax(logits, dim=-1)
        raise NotImplementedError



In [ ]:
def collect_episode_pg(env: QubitRotationEnv, policy: PolicyNet):
    obs = env.reset()
    log_probs, rewards, infos = [], [], []
    for _ in range(env.max_steps):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        probs = policy(obs_t)

        # TODO: sample action from Categorical(probs) and compute log_prob
        raise NotImplementedError

        obs, r, done, info = env.step(action)
        log_probs.append(log_prob)
        rewards.append(float(r))
        infos.append(info)
        if done:
            break
    return log_probs, rewards, infos

def discounted_returns(rewards, gamma: float = 0.99):
    # TODO: compute returns array (T,) using backward recursion
    raise NotImplementedError

def reinforce_update(optimizer, log_probs, returns_t: torch.Tensor):
    # TODO: loss = -(stack(log_probs) * returns_t).sum()
    # then backward + step
    raise NotImplementedError


In [ ]:
env = QubitRotationEnv(max_steps=40)
policy = PolicyNet(obs_dim=2, n_actions=env.n_actions).to(device)
opt = optim.Adam(policy.parameters(), lr=2e-3)

gamma = 0.99
episodes = 1200
ep_success = []

for ep in range(1, episodes+1):
    log_probs, rewards, infos = collect_episode_pg(env, policy)
    G = discounted_returns(rewards, gamma=gamma)
    returns_t = torch.tensor(G, dtype=torch.float32, device=device)

    # variance reduction
    returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

    # TODO: reinforce_update(opt, log_probs, returns_t)
    raise NotImplementedError

    final_F = infos[-1]["fidelity"]
    ep_success.append(1 if final_F >= env.success_F else 0)

    if ep % 200 == 0:
        print(f"PG ep {ep:4d} | success(last100)={np.mean(ep_success[-100:]):.2f} | finalF={final_F:.4f}")

# Evaluate deterministic argmax version
def act_pg_greedy(obs):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
    with torch.no_grad():
        probs = policy(obs_t)
    return int(torch.argmax(probs).item())

m_pg = run_episodes(env, act_pg_greedy, n_episodes=300)
print_metrics("PolicyGrad", m_pg)

# TODO: plot success rate curve (moving average)
raise NotImplementedError


---

# Part C — Q-learning with Neural Networks (DQN)

## What is learned?

We learn an **action–value function**
$
Q_\theta(s,a) \approx \mathbb{E}\Big[\sum_{t=0}^\infty \gamma^t r_t \,\Big|\, s_0=s,\; a_0=a,\; \text{optimal actions thereafter}\Big].
$

- Input: state \(s\)
- Output: one scalar **per action**
- Interpretation:
  > “How good is it to take action \(a\) in state \(s\), assuming optimal behavior afterwards?”

Unlike policy gradients, **we do not learn a policy directly**.  
The policy is obtained **implicitly** via:
$
\pi(s) = \arg\max_a Q_\theta(s,a).
$

---

## Bellman target (bootstrapping)

For a transition $((s,a,r,s',\text{done}))$, the **Bellman optimality equation** suggests:

$Q^*(s,a) = r + \gamma \max_{a'} Q^*(s',a').$

Since the true \(Q^\*\) is unknown, we construct a **training target**:

$
\boxed{
y = r + (1-\text{done})\,\gamma\,\max_{a'} Q_{\theta^-}(s',a')
}
$

- \(r\): immediate reward
- \($\gamma$\): discount factor (importance of future rewards)
- \($\max_{a'}$\): greedy assumption (optimal future behavior)
- \($(1-\text{done})$\): no future reward after terminal states
- \($\theta^-$\): parameters of a **target network**, held fixed for stability

This is called **bootstrapping**:  
we use the network’s *own predictions* to define the learning target.

---


## Loss function (Bellman error)

We train the network by minimizing the **squared Bellman error**:

$
\boxed{
\mathcal{L}(\theta) = \big(Q_\theta(s,a) - y\big)^2
}
$

This is a standard regression problem:
- input: \((s,a)\)
- target: \(y\)
- optimizer: stochastic gradient descent (Adam)

Each gradient step nudges \($Q_\theta(s,a)$\) toward satisfying the Bellman equation.

---

## Exploration vs exploitation

Although the learned policy is greedy,
$
a = \arg\max_a Q_\theta(s,a),
$
during training we use **ε-greedy exploration**:

- with probability \($\varepsilon$\): choose a random action
- with probability \($1-\varepsilon$\): choose \($\arg\max_a Q_\theta(s,a)$\)

This ensures sufficient exploration of the state–action space.

---

- Q-learning learns values, not probabilities
- Learning is off-policy (can learn from exploratory actions)

This makes DQN sample-efficient but delicate, in contrast to policy gradients, which are robust but noisier.


In [ ]:
class QNet(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buf = deque(maxlen=int(capacity))

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def __len__(self):
        return len(self.buf)

    def sample(self, batch_size: int):
        # TODO: sample random transitions and return tensors on device:
        # S (B,obs_dim), A (B,), R (B,), S2 (B,obs_dim), D (B,)
        raise NotImplementedError

def select_action_eps_greedy(qnet: QNet, obs, epsilon: float, n_actions: int):
    # TODO: with prob epsilon -> random action, else argmax Q
    raise NotImplementedError

def dqn_update(qnet, qnet_tgt, opt, batch, gamma: float = 0.99):
    S, A, R, S2, D = batch
    # TODO:
    # q_sa = qnet(S).gather(1, A[:,None]).squeeze(1)
    # y = R + (1-D)*gamma*max_a' qnet_tgt(S2)
    # loss = MSE(q_sa, y)
    # optimize
    raise NotImplementedError


In [ ]:
env = QubitRotationEnv(max_steps=40)
obs_dim = 2
n_actions = env.n_actions

qnet = QNet(obs_dim, n_actions).to(device)
qnet_tgt = QNet(obs_dim, n_actions).to(device)
qnet_tgt.load_state_dict(qnet.state_dict())

opt = optim.Adam(qnet.parameters(), lr=1e-3)
buf = ReplayBuffer(capacity=50000)

episodes = 1500
batch_size = 64
gamma = 0.99
warmup = 500
target_update_every = 400
eps_start, eps_end, eps_decay = 1.0, 0.05, 6000
step_count = 0

def epsilon_by_step(t):
    return eps_end + (eps_start - eps_end) * np.exp(-t / eps_decay)

success_hist = []

for ep in range(1, episodes+1):
    obs = env.reset()
    info = {"fidelity": env.fidelity(0.0)}
    for t in range(env.max_steps):
        eps = float(epsilon_by_step(step_count))
        action = select_action_eps_greedy(qnet, obs, eps, n_actions)

        next_obs, r, done, info = env.step(action)

        s_t = torch.tensor(obs, dtype=torch.float32, device=device)
        s2_t = torch.tensor(next_obs, dtype=torch.float32, device=device)
        buf.push(s_t, action, r, s2_t, done)

        obs = next_obs
        step_count += 1

        if len(buf) >= max(warmup, batch_size):
            batch = buf.sample(batch_size)
            # TODO: dqn_update
            pass

        if step_count % target_update_every == 0:
            qnet_tgt.load_state_dict(qnet.state_dict())

        if done:
            break

    success_hist.append(1 if info["fidelity"] >= env.success_F else 0)
    if ep % 250 == 0:
        print(f"DQN ep {ep:4d} | success(last200)={np.mean(success_hist[-200:]):.2f} | eps={eps:.3f} | finalF={info['fidelity']:.4f}")

def act_dqn_greedy(obs):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
    with torch.no_grad():
        q = qnet(obs_t.unsqueeze(0)).squeeze(0)
    return int(torch.argmax(q).item())

m_dqn = run_episodes(env, act_dqn_greedy, n_episodes=300)
print_metrics("DQN", m_dqn)


plt.figure()
plt.plot(moving_avg(success_hist, 100))
plt.xlabel("Episode"); plt.ylabel("Success rate (moving avg)")
plt.title("DQN: success")
plt.show()


---

## Video Visualize

In [ ]:
# If MP4 writing fails in Colab, run once:
!pip -q install imageio imageio-ffmpeg

import imageio.v2 as imageio
import base64
import IPython.display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Optional (nicer Bloch sphere):
!pip -q install qiskit


In [ ]:
def embed_mp4(filename: str, width=640, height=480):
    with open(filename, "rb") as f:
        video = f.read()
    b64 = base64.b64encode(video).decode()
    html = f"""
    <video width="{width}" height="{height}" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      Your browser does not support the video tag.
    </video>"""
    return IPython.display.HTML(html)

def fig_to_rgb_array(fig):
    """Convert Matplotlib figure to RGB array (backend-stable)."""
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba())
    return buf[:, :, :3]  # drop alpha

# Optional nicer Bloch plot via Qiskit
try:
    from qiskit.visualization import plot_bloch_vector as qiskit_plot_bloch_vector
    _HAS_QISKIT = True
except Exception:
    _HAS_QISKIT = False
    qiskit_plot_bloch_vector = None

def plot_bloch_vector_from_theta(theta: float):
    """Bloch vector for this family: (sin theta, 0, cos theta)."""
    x = float(np.sin(theta)); y = 0.0; z = float(np.cos(theta))
    if _HAS_QISKIT:
        fig = qiskit_plot_bloch_vector([x, y, z], title=f"theta={theta:.3f}")
        fig.set_size_inches(5, 4, forward=True)
        plt.tight_layout()
        return fig

    fig = plt.figure(figsize=(5, 4))
    ax = fig.add_subplot(111, projection="3d")

    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_surface(xs, ys, zs, linewidth=0, alpha=0.12)

    ax.plot([-1, 1], [0, 0], [0, 0], linewidth=1)
    ax.plot([0, 0], [-1, 1], [0, 0], linewidth=1)
    ax.plot([0, 0], [0, 0], [-1, 1], linewidth=1)

    ax.quiver(0, 0, 0, x, y, z, length=1.0, normalize=False, linewidth=2)

    ax.set_xlim([-1, 1]); ax.set_ylim([-1, 1]); ax.set_zlim([-1, 1])
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"theta={theta:.3f}")
    ax.view_init(elev=22, azim=35)
    plt.tight_layout()
    return fig


In [ ]:
def create_qubit_video_from_act_fn(act_fn, filename="policy", fps=6, max_steps=100):
    filename = filename + ".mp4"
    env = QubitRotationEnv(max_steps=max_steps)
    obs = env.reset()
    frames = []

    fig = plot_bloch_vector_from_theta(theta=0.0)
    frames.append(fig_to_rgb_array(fig)); plt.close(fig)

    info = {"theta": 0.0}
    for _ in range(max_steps):
        action = int(act_fn(obs))
        obs, r, done, info = env.step(action)

        fig = plot_bloch_vector_from_theta(theta=info["theta"])
        frames.append(fig_to_rgb_array(fig)); plt.close(fig)

        if done:
            break

    with imageio.get_writer(filename, fps=fps) as video:
        for fr in frames:
            video.append_data(fr)

    return embed_mp4(filename)



In [ ]:
from IPython.display import display

display(create_qubit_video_from_act_fn(act_random, filename="random-policy"))
display(create_qubit_video_from_act_fn(act_pg_greedy, filename="pg-policy"))
display(create_qubit_video_from_act_fn(act_dqn_greedy, filename="dqn-policy"))